## Murmuration of Elliptic Curves 

This IPython (or, Jupyter) notebook generates the result from [HLOP24]. 

The paper [HLOP24] reports the _murmuration_ property of the elliptic curves. One of the result they report can be vaguely described as:

_"The Frobenius traces $(a_p)_{p \in\{2,3,\dots, p_n\}}$ can somehow encode the data of elliptic curve $E$, e.g. its rank $r_E$."_

Actually one can say further that there exist some oscillating property of the average of $a_p$ over the class of elliptic curves which ranges over a fixed window of conductors as follows. 

![title](murmuration_10000_30000_and_1000_primes.png)

The blue and orange points represent the average of $a_p$ over the elliptic curves of conductor in $[10000, 30000]$ and of rank 0, 1, respectively. 

[HLOP24]: He, Y. H., Lee, K. H., Oliver, T., & Pozdnyakov, A. (2024). Murmurations of Elliptic Curves. Experimental Mathematics, 1–13. https://doi.org/10.1080/10586458.2024.2382361

## Prerequisite

### SageMath
The calculation is done in two steps; one is on the `SageMath` and the other is on the Python. You need both `SageMath` and `Python 3`.

This notebook runs with `SageMath 9.2` and `Python 3.10.13`; other Python 3 versions would be also compatible. 

Also, you need some python packages like `numpy`, `pandas`, `matplotlib`.

My environment is:
* `MacOS 15.2` (Sequoia),
* `SageMath` version `9.2` using `Python 3.8.5`,
* `Python 3.10.0` using `pip 21.3.1`.

### Cremona Database
You might need to install the Cremona database for your Sagemath. 

For example, on MacOS, the following command on the terminal will work: 

`$ sage -i database_cremona_ellcurve`

refer to the following [link on the database](https://doc.sagemath.org/html/en/reference/databases/sage/databases/cremona.html
).

### Data from LMFDB 
Frobeinus traces $a_p$ is computed in this notebook, but the label of the curves and the rank data can be obtained from LMFDB. 

1. Visit [this link](https://www.lmfdb.org/EllipticCurve/Q/?conductor=10000-99999).
2. Search the elliptic curves for the range of conductors you want.
3. Check `one` from the `Curves per isogeny class` dropdown menu.
4. Select the followings from the dropdown menu `Select` and download. You might need to click `determine the number of results` and wait for a while. 
    - `LMFDB curve label`: LMFDB label for each elliptic curve
    - `LMFDB class label`: isogeny class. Note that $E$ and $E'$ has the same rank if they are isogenous, and [HLOP24] uses the isogeny classes (not the isomorphism classes) to see the murmurations. 
    - `conductor`
    - `rank`
5. Locate the downloaded file; it should be a `txt` file.

## Data preparation and preprocessing

In [1]:
import os 
import time 
import pickle
import numpy as np
import pandas as pd 

In [93]:
# As the data can be large, I recommend to divide the 
#   data in smaller bins and run the code for each of
#   them. If your machine has multicore CPUs and
#   enough memory, you can run them in parallel.
#   (In-code parallel computing was not implemented.)

# For example, the number of isogeny classes of
#   elliptic curves with conductor in [1, 10^5) are
#   about 437,226, so there are 8 bins with binsize
#   60,000; you can make run this notebook 8 times
#   (or make some copies of this notebook and run
#   simultaneously) for each of them, varying the
#   bin_idx from 1 to 8. 
#
# You can choose your own binsize you like, or choose
#   a big enough binsize to run the code for the whole
#   data at once.

binsize = 60000
bin_idx = 1 

In [94]:
# Open the txt file downloaded from LMFDB
with open('data/lmfdb_ec_curvedata_0110_0830.txt', 'r') as f:
    iter_line = iter(f)
    table = [line.strip().split('\t')
             for line in iter_line if line.strip() and not line.startswith('#')]

In [95]:
df = pd.DataFrame(
    table[binsize*(bin_idx-1):binsize*(bin_idx)],
    columns = ['label', 'isogeny_class', 'conductor', 'rank'])
del(table)

# The rank are not big. (Largest known one for the
#   elliptic curves over Q is 29 for now.) Thus
#   np.int8 is enough. 
#
# The conductur are less than 100,000, so we can use
#   np.int64. You can change it if you need.
# 
# Be sure that you don't choose too big or too small. 

df['conductor'] = df['conductor'].astype(np.int64)
df['rank'] = df['rank'].astype(np.int8)

# The labels has quotation marks so we remove them. 
df['label'] = df['label'].str[1:-1]
df['isogeny_class'] = df['isogeny_class'].str[1:-1]

# If you have some column you don't want, you can
#   first name it as above and then drop it like 
# df = df.drop(columns=['torsion'])


In [ ]:
# One can check the dataframe. You should see the
#   unlabelled index column ranges from 0 to
#   min((binsize-1), (# of the dataset)).
df

In [ ]:
# Select the first num_p primes.
# The murmuration pattern can be observed with a
#   balanced size of num_p and n_max/n_min. (num_p 
#   shouldn't be too small relative to n_max/n_min)
#
# Data can be obtained for a larger range and then 
#   reduced for a smaller range.
#
# For the converse (i.e. to expand the data):
# Expanding your previous calculation result to the
#   conductors with the same num_p can be done by 
#   simply appending the dataframe. 
# However, it is not efficient to run the code for the
#   same conductor window with a larger num_p again to
#   expand the data. (SageMath always calculates the
#   aplist from p=2 to the given p) Therefore, if you
#   need data for a large num_p and your device has 
#   limited computational power, it would be wiser to
#   start with a large num_p and a smaller conductor 
#   window, and then expand the conductors afterwards.

num_p = 5000 

# first num_p primes.
primes = prime_range(1, Primes().unrank(num_p))
largest_prime = primes[-1]

# Store the primes as a pickle file.
with open('processed_data/5000_primes.pkl', 'wb') as f:
    pickle.dump([int(p) for p in primes], f)

print(len(primes), 'primes, with the largest prime =', largest_prime)

In [ ]:
# We now calculate the Frobenius traces ap's for the
#   first num_p prime p's with SageMath function. 

# The nparray has num_p columns and len(df) rows, so
#   we i`nitialize the numpy array with the shape
#   (len(df), num_p).

# Note that the Deligne's bound $a_p\leq2\sqrt{p}$
#   allow us to use np.int16. You might want to use
#   bigger dtype if you are dealing with much larger
#   num_p.

ap_array = np.zeros((len(df), num_p), dtype=np.int16)

# Now we update the ap_array for each row of df. For each label, the row should be:
# EllipticCurve(label).aplist(Primes()[num_p-1])

timenow = time.time()
for idx, row in df.iterrows():
    ap_array[idx] = EllipticCurve(
        row['label']).aplist(largest_prime)
    if idx %5000 ==0 and idx>0:
        print('{} elliptic curves has been covered.\n{:.2f}% done.'.format(idx, float(idx/len(df)*100)))
        
        timedelta = time.time() - timenow
        print(f'{timedelta:.2f}s passed for this batch')
            
        print() 
        timenow = time.time()

In [ ]:
# Now we should append the ap_array to the df.
# each column names should be 'a'+str(p) for p in primes.
for idx, p in enumerate(primes):
    df['a'+str(p)] = ap_array[:, idx]

In [ ]:
# Check if the dataframe is correctly updated.
df 

In [ ]:
# serialize the dataframe 
file_path = './dataframes/df_{}_1to100000_primeindex5000.pkl'.format(bin_idx)
with open(file_path, 'wb') as f:
    pickle.dump(df, f)